# NB-11 — Audit & release (revision work packages WP1, WP2, WP5-evidence, WP12-prep, WP13-prep, Zenodo prep)

**Question this notebook answers:** *do the numbers in the submitted paper reproduce with the code and checkpoints we
actually have — and what did the reviewers (and we) not yet know about our own pipeline?*
The answer decides whether the contingency notebook NB-18 is needed. **No model is trained here** (inference only).

What it produces (all in `/kaggle/working`):

| Output | Used by |
|---|---|
| `per_chip_counts.csv.gz` — TP/FP/FN/TN per chip × regime × model × seed × split (zero-fill, plus mean-fill on test) | **NB-16** (statistics: paired differences, event-blocked bootstrap), NB-17 (figures) |
| `audit_report.md`, `audit_summary.json` — every check below with a pass / check / mismatch verdict | response letter, WP1/WP5 text |
| `table2_reproduction.csv`, `all_results_matrix.csv`, `bolivia_results.csv` | Table 2 / Table 4 / Section 4.7 rebuilds |
| `release/` — split manifest, norm stats, checkpoint hashes, EMA-only weights, code snapshot, env, Zenodo README draft | Editor's code-in-a-DOI-repository requirement |

**Attach:** `lgmf_v7` (the code; do not attach lgmf_v2-v6) · Sen1Floods11 · `lgmf-index` · every checkpoint dataset (`lgmf-segf`, `lgmf-baselines`, the NB-final output, and optionally `lgmf-sweep`, inject / v6 outputs).
**Settings:** GPU T4 · Internet **On** (smp/thop pip fallbacks) · *Save & Run All (Commit)*, then turn the output into the dataset `lgmf-nb11-audit`. ~30-40 min.

## 0. Bootstrap

In [ ]:
import sys, os, glob, json, time, hashlib, platform, shutil, warnings, dataclasses, copy
warnings.filterwarnings("ignore")
import re, importlib
# 1) drop any lgmf already imported in this kernel (a stale import from a wrong copy causes "No module named lgmf.baselines")
for _m in [m for m in sys.modules if m == "lgmf" or m.startswith("lgmf.")]: del sys.modules[_m]
importlib.invalidate_caches()
REQUIRED = ["__init__.py", "config.py", "dataset.py", "model.py", "baselines.py", "engine.py", "curriculum.py", "metrics.py", "losses.py", "ema.py"]
def _ver(p):
    m = re.search(r'__version__\s*=\s*"([\d.]+)"', open(p).read())
    return tuple(int(x) for x in m.group(1).split(".")) if m else (0,)
_all = sorted(glob.glob("/kaggle/input/**/lgmf/__init__.py", recursive=True))
print("every lgmf/ folder found under /kaggle/input:")
_cands = []
for p in _all:
    d = os.path.dirname(p); missing = [f for f in REQUIRED if not os.path.exists(os.path.join(d, f))]
    print(f"   v{'.'.join(map(str, _ver(p)))}  {'COMPLETE' if not missing else 'missing ' + ','.join(missing)}  {d}")
    if not missing: _cands.append((_ver(p), d))
assert _cands, "no complete lgmf package found -- attach the lgmf_v7 dataset"
_best = max(_cands)
assert _best[0] >= (7,), f"best complete lgmf is v{_best[0]}; attach lgmf_v7"
PKG_DIR = _best[1]
sys.path = [os.path.dirname(PKG_DIR)] + [q for q in sys.path if q != os.path.dirname(PKG_DIR)]
print("using", PKG_DIR)
for mod, pipname in (("segmentation_models_pytorch", "segmentation-models-pytorch"), ("thop", "thop")):
    try: __import__(mod)
    except ImportError: os.system(f"pip install -q {pipname}")
import numpy as np, pandas as pd, torch, rasterio
from rasterio.warp import transform_bounds
from torch.utils.data import DataLoader
import lgmf
assert os.path.dirname(lgmf.__file__) == PKG_DIR, f"wrong lgmf imported: {lgmf.__file__}"
from lgmf.config import LGMFConfig
from lgmf.dataset import Sen1Floods11, _read_raster
from lgmf.baselines import build_baseline, build_gated_unet
from lgmf.model import build_model, count_params
from lgmf.metrics import flops_g
from lgmf.engine import evaluate
from lgmf.curriculum import eval_regimes, apply_present, regime_probs
print("lgmf", lgmf.__version__)

ns = sorted(glob.glob("/kaggle/input/**/norm_stats.json", recursive=True))
assert ns, "attach lgmf-index"
if len(ns) > 1: print("WARNING: several norm_stats.json found, using the first:", ns)
idx = os.path.dirname(ns[0]); stats = json.load(open(ns[0]))
SPLITS = ["train", "val", "test", "bolivia"]
for s in SPLITS: assert os.path.exists(f"{idx}/{s}_index.json"), f"missing {s}_index.json in {idx}"

OUT = "/kaggle/working"; REL = f"{OUT}/release"; os.makedirs(REL, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "(CPU - slow; use a GPU session)")
REGS = ["both", "s1_only", "s2_only"]
SUMMARY = {}          # filled by every section, dumped to audit_summary.json
REPORT_LINES = []     # filled by every section, dumped to audit_report.md
def note(s): print(s); REPORT_LINES.append(s)

## 1. Pure helpers (numpy / pandas only)
Kept free of torch so they can be unit-tested offline. The self-tests at the bottom run every time the notebook runs.

In [ ]:
EPS = 1e-6                                            # identical to lgmf.metrics.SegMeter
def event_of(cid):  return cid.split("_")[0]          # 'Bolivia_103757' -> 'Bolivia'

def metrics_from_counts(tp, fp, fn, tn):
    tp, fp, fn, tn = (np.asarray(x, dtype=float) for x in (tp, fp, fn, tn))
    iou = tp / (tp + fp + fn + EPS)
    prec = tp / (tp + fp + EPS); rec = tp / (tp + fn + EPS)
    f1 = 2 * prec * rec / (prec + rec + EPS)
    return dict(iou=iou, f1=f1, precision=prec, recall=rec)

KEYS = ["config", "seed", "split", "variant", "regime"]
def pooled(df, by, flood=False):
    """Pool confusion counts over chips (this is how SegMeter / the paper computes IoU)."""
    cols = ["ftp", "ffp", "ffn", "ftn"] if flood else ["tp", "fp", "fn", "tn"]
    g = df.groupby(by, dropna=False)[cols].sum()
    out = g.copy(); out.columns = ["tp", "fp", "fn", "tn"]
    for k, v in metrics_from_counts(*[g[c].values for c in cols]).items(): out[k] = v
    return out.reset_index()

def across_seeds(ps, value="iou", by=("config", "split", "variant", "regime")):
    by = list(by)
    g = ps.groupby(by, dropna=False)[value]
    return pd.DataFrame({"n": g.size(), "mean": g.mean(),
                         "std0": g.apply(lambda x: float(np.std(x.values, ddof=0))),
                         "std1": g.apply(lambda x: float(np.std(x.values, ddof=1)) if len(x) > 1 else np.nan)}).reset_index()

def parse_ckpt_path(p):
    sd = os.path.dirname(p); seed_dir = os.path.basename(sd); cfg = os.path.basename(os.path.dirname(sd))
    seed = int(seed_dir[4:]) if seed_dir.startswith("seed") and seed_dir[4:].isdigit() else None
    return cfg, seed

def boxes_touch(a, b, tol):
    """a, b = (xmin, ymin, xmax, ymax); True if they overlap or are within tol of each other."""
    return not (a[2] + tol < b[0] or b[2] + tol < a[0] or a[3] + tol < b[1] or b[3] + tol < a[1])

def touching_ref(target_ids, ref_ids, boxes, tol):
    """For each target chip: list of reference chips whose footprint touches it."""
    return {t: [r for r in ref_ids if r != t and boxes_touch(boxes[t], boxes[r], tol)] for t in target_ids}

def collapse_label(tp, fp, fn, tn):
    tot = max(tp + fp + fn + tn, 1.0); predf = (tp + fp) / tot; rec = tp / (tp + fn + EPS)
    if predf > 0.95 and rec > 0.99: return "ALL-POSITIVE"
    if predf < 0.001:               return "ALL-NEGATIVE"
    return ""

def sched_probs(epoch, epochs, warm, end, start=(0.8, 0.1, 0.1)):
    if epoch < warm: return (1.0, 0.0, 0.0)
    t = min((epoch - warm) / max(epochs - warm, 1), 1.0)
    return tuple(s + t * (e - s) for s, e in zip(start, end))

def exposure(epochs, warm, end):
    arr = np.array([sched_probs(e, epochs, warm, end) for e in range(epochs)])
    return arr.mean(0), arr[-1]

# ---- the numbers printed in the submitted paper (Table 2, test split). (mean, std) with std = np.std(ddof=0).
PUBLISHED = {
 "segformer_concat":    dict(both=(.810,.002), s1_only=(.643,.010), s2_only=(.802,.008), flood=(.523,.012)),
 "segformer_noconsist": dict(both=(.781,.016), s1_only=(.606,.027), s2_only=(.773,.007), flood=(.483,.031)),
 "unet_drop":           dict(both=(.814,.002), s1_only=(.626,.012), s2_only=(.817,.007), flood=(.502,.015)),
 "manet_drop":          dict(both=(.737,.091), s1_only=(.564,.047), s2_only=(.724,.110), flood=(.441,.049)),
 "unet_naive":          dict(both=(.753,.064), s1_only=(.327,.214), s2_only=(.619,.035), flood=(.230,.158)),
}
LABEL = {"segformer_concat": "SegFormer + curriculum + consistency", "segformer_noconsist": "SegFormer + curriculum",
         "unet_drop": "U-Net + curriculum", "manet_drop": "MANet + curriculum", "unet_naive": "U-Net, no curriculum"}

def compare_to_published(measured, published, tol_ok=0.005, tol_bad=0.02):
    """measured[(config, metric)] = dict(mean, std0, std1, n)."""
    rows = []
    for cfg, regs in published.items():
        for met, (pm, ps) in regs.items():
            m = measured.get((cfg, met))
            if m is None:
                rows.append(dict(config=cfg, metric=met, published=pm, measured=np.nan, delta=np.nan, status="NOT EVALUATED")); continue
            d = m["mean"] - pm
            st = "OK" if abs(d) <= tol_ok else ("CHECK" if abs(d) <= tol_bad else "MISMATCH")
            rows.append(dict(config=cfg, metric=met, published=pm, measured=round(m["mean"], 4), delta=round(d, 4),
                             pub_std=ps, std0=round(m["std0"], 4), std1=round(m["std1"], 4) if m["std1"] == m["std1"] else np.nan,
                             n_seeds=m["n"], status=st))
    return pd.DataFrame(rows)

# ---- self-tests (run on every execution) ----
_m = metrics_from_counts([50], [25], [25], [900]); assert abs(float(_m["iou"][0]) - 0.5) < 1e-4
_pc = pd.DataFrame({"config": ["a"]*4, "seed": [1, 1, 2, 2], "split": ["t"]*4, "variant": ["z"]*4, "regime": ["both"]*4,
                    "tp": [10, 30, 20, 20], "fp": [0, 0, 10, 10], "fn": [10, 10, 0, 0], "tn": [0]*4,
                    "ftp": [1]*4, "ffp": [1]*4, "ffn": [1]*4, "ftn": [1]*4})
_ps = pooled(_pc, KEYS); assert np.allclose(_ps["iou"], [2/3, 2/3], atol=1e-4), _ps       # pooled (0.667), not mean-of-chips (0.625)
_ag = across_seeds(_ps); assert _ag["n"].iloc[0] == 2 and abs(_ag["std1"].iloc[0] - _ag["std0"].iloc[0] * np.sqrt(2)) < 1e-9
assert parse_ckpt_path("/x/ckpt_segf/segformer_concat/seed42/best.pt") == ("segformer_concat", 42)
assert boxes_touch((0, 0, 1, 1), (1.00001, 0, 2, 1), 0.001) and not boxes_touch((0, 0, 1, 1), (1.5, 0, 2, 1), 0.001)
assert collapse_label(100, 900, 0, 0) == "ALL-POSITIVE" and collapse_label(0, 0, 100, 900) == "ALL-NEGATIVE" and collapse_label(50, 10, 10, 930) == ""
_e, _l = exposure(45, 5, (0.4, 0.45, 0.15))
assert np.allclose(_e, [0.6489, 0.2406, 0.1106], atol=5e-4) and np.allclose(_l, [0.41, 0.4412, 0.1488], atol=5e-4), (_e, _l)
print("helper self-tests passed")

## 2. Split audit (WP: leakage disclosure)
Events are derived from chip-id prefixes (`Bolivia_103757` → `Bolivia`). Checks: no chip in two partitions; chip counts match what the paper reports
(250 / 89 / 90 / 15); how many events straddle partitions; and — stronger than event labels — how many **val/test chips physically touch a training chip**.

In [ ]:
IDX = {s: json.load(open(f"{idx}/{s}_index.json")) for s in SPLITS}
SM = pd.DataFrame([dict(id=it["id"], split=s, event=event_of(it["id"])) for s in SPLITS for it in IDX[s]])
cnt = SM.groupby("split").size().reindex(SPLITS)
note("### Split sizes\n" + ", ".join(f"{s}={cnt[s]}" for s in SPLITS) + "  (paper: train 250 / val 89 / test 90 / bolivia 15)")
SUMMARY["split_sizes"] = {s: int(cnt[s]) for s in SPLITS}
SUMMARY["split_sizes_match_paper"] = bool(tuple(cnt[s] for s in SPLITS) == (250, 89, 90, 15))
dup = SM[SM.duplicated("id", keep=False)]
note(f"chips present in more than one partition: {dup['id'].nunique()}  (must be 0)")
SUMMARY["chips_in_multiple_partitions"] = int(dup["id"].nunique())
ev = SM.pivot_table(index="event", columns="split", values="id", aggfunc="count", fill_value=0).reindex(columns=SPLITS, fill_value=0)
ev["partitions"] = (ev[["train", "val", "test", "bolivia"]] > 0).sum(axis=1)
display(ev)
shared = ev[(ev[["train", "val", "test"]] > 0).sum(axis=1) > 1]
note(f"events appearing in >1 of train/val/test: {len(shared)} of {len(ev)}  ->  {list(shared.index)}")
note(f"test chips whose EVENT also has training chips: {int(SM[(SM.split=='test') & SM.event.isin(SM[SM.split=='train'].event)].shape[0])} / {int(cnt['test'])}")
SUMMARY["events_total"] = int(len(ev)); SUMMARY["events_shared_train_val_test"] = int(len(shared))
SUMMARY["test_chips_event_in_train"] = int(SM[(SM.split=='test') & SM.event.isin(SM[SM.split=='train'].event)].shape[0])
note(f"train chips dropped by the >98% no-data filter: {stats.get('dropped_train_chips')}")

In [ ]:
# physical adjacency: footprints reprojected to EPSG:4326; "touching" = overlap or gap < ~30 m
BOX = {}
for s in SPLITS:
    for it in IDX[s]:
        with rasterio.open(it["label"]) as src:
            BOX[it["id"]] = transform_bounds(src.crs, "EPSG:4326", *src.bounds)
TOL = 0.0003
train_ids = [it["id"] for it in IDX["train"]]
ADJ = {}
for s in ("val", "test", "bolivia"):
    ids = [it["id"] for it in IDX[s]]
    t = touching_ref(ids, train_ids, BOX, TOL)
    n_touch = sum(1 for v in t.values() if v)
    ADJ[s] = dict(chips=len(ids), touching_train=n_touch, frac=round(n_touch / max(len(ids), 1), 3),
                  mean_touching_neighbours=round(float(np.mean([len(v) for v in t.values()])), 2))
    SM.loc[SM.id.isin(ids), "touches_train"] = SM.loc[SM.id.isin(ids), "id"].map(lambda i: len(t[i]) > 0)
note("### Physical adjacency to training chips\n" + "\n".join(f"- {s}: {a['touching_train']}/{a['chips']} chips touch a training chip ({a['frac']:.0%}); mean neighbours {a['mean_touching_neighbours']}" for s, a in ADJ.items()))
SUMMARY["adjacency_to_train"] = ADJ
SM["bbox_wsen"] = SM.id.map(lambda i: [round(v, 6) for v in BOX[i]])
SM.to_csv(f"{REL}/split_manifest.csv", index=False)

## 3. Cache evaluation data + imputation audit (Reviewer 2.11: "mean imputation" vs the code)
`dataset.py` replaces non-finite pixels with **0.0 in raw dB/reflectance, before standardisation**. For S1 (mean ≈ −11 dB) that standardises to ≈ +2-3σ — neither zero nor the band mean.
Below: how many pixels this touches, and the actual test effect of switching to true mean-fill (= 0 after standardisation) at inference.

In [ ]:
base = LGMFConfig()
base.s1_mean, base.s1_std = stats["s1_mean"], stats["s1_std"]
base.s2_mean, base.s2_std = stats["s2_mean"], stats["s2_std"]; base.num_workers = 0
fill1 = (0.0 - np.array(stats["s1_mean"])) / (np.array(stats["s1_std"]) + 1e-6)
fill2 = (0.0 - np.array(stats["s2_mean"])) / (np.array(stats["s2_std"]) + 1e-6)
note("### Imputation (code behaviour)\n"
     f"- S1 fill value after standardisation (VV, VH): {np.round(fill1, 2).tolist()} sigma  (mean-fill would be 0)\n"
     f"- S2 fill value after standardisation (6 bands): {np.round(fill2, 2).tolist()} sigma  (clamped to +-10)")
SUMMARY["s1_fill_std_value"] = np.round(fill1, 3).tolist(); SUMMARY["s2_fill_std_value"] = np.round(fill2, 3).tolist()

class Cache:
    """Whole split in RAM (float32, ~11 MB/chip). Also keeps masks of non-finite raw pixels."""
    def __init__(self, split):
        ds = Sen1Floods11(base, f"{idx}/{split}_index.json", augment=False)
        self.split, self.ids = split, [it["id"] for it in ds.items]
        self.s1, self.s2, self.label, self.valid, self.jrc, self.nf1, self.nf2, self.stat = [], [], [], [], [], [], [], []
        self.missing_jrc = 0
        for i, it in enumerate(ds.items):
            d = ds[i]
            raw1 = _read_raster(it["s1"]); raw2 = _read_raster(it["s2"], list(base.s2_bands))
            self.nf1.append(torch.from_numpy(~np.isfinite(raw1))); self.nf2.append(torch.from_numpy(~np.isfinite(raw2)))
            self.stat.append(dict(split=split, id=it["id"], s1_nonfinite=float((~np.isfinite(raw1)).mean()),
                                  s1_exact_zero=float((np.nan_to_num(raw1, nan=1.0) == 0).mean()),
                                  s2_nonfinite=float((~np.isfinite(raw2)).mean()), nodata_label=float(1 - d["valid"].mean())))
            self.s1.append(d["s1"]); self.s2.append(d["s2"]); self.label.append(d["label"]); self.valid.append(d["valid"])
            if "jrc" in d: self.jrc.append(d["jrc"])
            else: self.jrc.append(torch.zeros_like(d["label"])); self.missing_jrc += 1
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        return dict(s1=self.s1[i], s2=self.s2[i], label=self.label[i], valid=self.valid[i], jrc=self.jrc[i])
    def batch(self, ii, variant="zero"):
        st = lambda L: torch.stack([L[i] for i in ii])
        b = dict(s1=st(self.s1), s2=st(self.s2), label=st(self.label), valid=st(self.valid), jrc=st(self.jrc))
        if variant == "mean":      # true mean imputation: non-finite pixels -> 0 AFTER standardisation
            b["s1"] = b["s1"].masked_fill(st(self.nf1), 0.0); b["s2"] = b["s2"].masked_fill(st(self.nf2), 0.0)
        return b

t0 = time.time()
CACHE = {s: Cache(s) for s in ("val", "test", "bolivia")}
print({s: len(c) for s, c in CACHE.items()}, f"cached in {time.time()-t0:.0f}s; chips missing JRC:", {s: c.missing_jrc for s, c in CACHE.items()})
# fidelity check: cached tensors must equal the dataset's own output
_d = Sen1Floods11(base, f"{idx}/test_index.json", augment=False)[0]
assert torch.equal(_d["s1"], CACHE["test"].s1[0]) and torch.equal(_d["s2"], CACHE["test"].s2[0])
IMP = pd.DataFrame([r for c in CACHE.values() for r in c.stat])
# train split: raw read only (not cached)
tr_rows = []
for it in IDX["train"]:
    r1 = _read_raster(it["s1"]); r2 = _read_raster(it["s2"], list(base.s2_bands))
    tr_rows.append(dict(split="train", id=it["id"], s1_nonfinite=float((~np.isfinite(r1)).mean()),
                        s1_exact_zero=float((np.nan_to_num(r1, nan=1.0) == 0).mean()), s2_nonfinite=float((~np.isfinite(r2)).mean())))
IMP = pd.concat([IMP, pd.DataFrame(tr_rows)], ignore_index=True)
imp_tab = IMP.groupby("split").agg(chips=("id", "count"),
            chips_with_nonfinite_S1=("s1_nonfinite", lambda x: int((x > 0).sum())),
            pct_S1_nonfinite=("s1_nonfinite", lambda x: 100 * x.mean()),
            max_pct_S1_chip=("s1_nonfinite", lambda x: 100 * x.max()),
            chips_with_nonfinite_S2=("s2_nonfinite", lambda x: int((x > 0).sum())),
            pct_S1_exact_zero=("s1_exact_zero", lambda x: 100 * x.mean())).reindex(SPLITS).round(4)
display(imp_tab); imp_tab.to_csv(f"{OUT}/imputation_audit.csv")
SUMMARY["imputation"] = imp_tab.reset_index().to_dict("records")

## 4. Checkpoint discovery + robust model rebuild
Gated checkpoints do not store `arch`/`encoder` (the old `build_from_ckpt` could not rebuild them) — here the architecture is inferred from the run name and **verified by a strict `load_state_dict`**, so a wrong guess cannot pass silently.

In [ ]:
def discover():
    ps = sorted(set(glob.glob("/kaggle/input/**/best.pt", recursive=True) + glob.glob("/kaggle/working/**/best.pt", recursive=True)))
    rows = []
    for p in ps:
        cfg, seed = parse_ckpt_path(p); rows.append(dict(path=p, config=cfg, seed=seed))
    return pd.DataFrame(rows)
CK_ALL = discover(); assert len(CK_ALL), "no best.pt found -- attach the checkpoint datasets"
dups = CK_ALL[CK_ALL.duplicated(["config", "seed"], keep=False)].sort_values(["config", "seed"])
CK = CK_ALL.drop_duplicates(["config", "seed"], keep="first").reset_index(drop=True)
print(f"{len(CK_ALL)} checkpoints found, {len(CK)} unique (config, seed); duplicates: {len(dups)-dups[['config','seed']].drop_duplicates().shape[0]}")
if len(dups): display(dups)
display(CK.groupby("config").seed.apply(lambda x: sorted(x.dropna().astype(int).tolist())).to_frame("seeds"))

ARCHS = ["unet", "manet", "fpn", "linknet", "deeplabv3plus", "pspnet", "unetplusplus", "segformer", "upernet"]
ENCS  = ["mobilenet_v2", "mit_b0", "resnet18", "resnet34", "efficientnet-b0", "timm-mobilenetv3_large_100"]
def arch_candidates(name):
    n = name.lower()
    a = [x for x in ARCHS if x in n] or ARCHS
    e = [x for x in ENCS if x.replace("-", "").replace("_", "") in n.replace("-", "").replace("_", "")] + ENCS
    e = list(dict.fromkeys(e))
    if "segformer" in n: e = ["mit_b0"] + [x for x in e if x != "mit_b0"]
    return [(x, y) for x in a for y in e]

def build_from_ckpt(ck, name):
    sd = ck.get("ema") or ck["model"]
    if ck.get("gated"):
        g = dict(ck["gated"]); out_ch = g.pop("out_ch", 8); a0, e0 = g.pop("arch", None), g.pop("encoder", None)
        cands = [(a0, e0)] if a0 and e0 else arch_candidates(name)
        last = None
        for a, e in cands:
            try:
                m = build_gated_unet(arch=a, encoder=e, out_ch=out_ch, pretrained=False, **g); m.load_state_dict(sd); return m, (a, e, "gated")
            except Exception as ex: last = ex
        raise RuntimeError(f"gated rebuild failed for {name}: {str(last)[:160]}")
    if ck.get("baseline"):
        kw = dict(ck["baseline"]); in_ch = kw.pop("in_ch", 8); a0, e0 = kw.get("arch"), kw.get("encoder")
        cands = [(a0, e0)] if a0 and e0 else arch_candidates(name)
        last = None
        for a, e in cands:
            try:
                m = build_baseline(arch=a, encoder=e, in_ch=in_ch, pretrained=False); m.load_state_dict(sd); return m, (a, e, "concat")
            except Exception as ex: last = ex
        raise RuntimeError(f"baseline rebuild failed for {name}: {str(last)[:160]}")
    fields = {f.name for f in dataclasses.fields(LGMFConfig)}
    c = LGMFConfig(**{k: v for k, v in ck["cfg"].items() if k in fields}); c.s2_bands = tuple(c.s2_bands); c.encoder_pretrained = False
    m = build_model(c); m.load_state_dict(sd); return m, ("lgmf", c.encoder_name, "lgmf")

def safe_flops(m):
    try: return float(flops_g(copy.deepcopy(m).cpu(), LGMFConfig(), device="cpu"))
    except Exception as ex: return -1.0

## 5. Per-chip evaluation engine
Confusion counts are produced **per chip, per regime**, for both the full valid mask and the flood-only mask (JRC permanent water removed). Pooling them reproduces `SegMeter` exactly — verified below against `lgmf.engine.evaluate` on a real checkpoint (assertion).
The per-chip file is also the input NB-16 needs for paired / event-blocked statistics.

In [ ]:
@torch.no_grad()
def chip_counts(model, cache, ctx, variant="zero", bs=8):
    model.eval(); rows = []; R = eval_regimes()
    for s in range(0, len(cache), bs):
        ii = list(range(s, min(s + bs, len(cache)))); b = cache.batch(ii, variant)
        s1, s2 = b["s1"].to(device), b["s2"].to(device)
        tgt, valid, jrc = b["label"].to(device), b["valid"].to(device), b["jrc"].to(device)
        for reg, basep in R.items():
            present = basep.repeat(len(ii), 1).to(device)
            s1i, s2i = apply_present(s1, s2, present)
            pred = (torch.sigmoid(model(s1i, s2i, present)["logits"]).squeeze(1) > 0.5).float()
            out = {}
            for tag, m in (("", valid), ("f", valid * (1.0 - jrc))):
                p, t = pred * m, tgt * m
                out[tag + "tp"] = (p * t).sum((1, 2)); out[tag + "fp"] = (p * (1 - t) * m).sum((1, 2))
                out[tag + "fn"] = ((1 - p) * t).sum((1, 2)); out[tag + "tn"] = ((1 - p) * (1 - t) * m).sum((1, 2))
            for k, i in enumerate(ii):
                rows.append(dict(**ctx, variant=variant, regime=reg, id=cache.ids[i], event=event_of(cache.ids[i]),
                                 **{c: float(v[k]) for c, v in out.items()}))
    return rows
print("engine defined")

## 6. Evaluate every checkpoint (val / test / bolivia, zero-fill as in training) — plus mean-fill on test for the paper's models

In [ ]:
REPORT = list(PUBLISHED.keys())
EVAL_ALL = True            # False -> evaluate only the five Table-2 configs
rows, META, ARCH_OF, FAIL, equiv_done = [], {}, {}, [], False
t0 = time.time()
for r in CK.itertuples():
    if not EVAL_ALL and r.config not in REPORT: continue
    try:
        ck = torch.load(r.path, map_location="cpu", weights_only=False)
        m, arch = build_from_ckpt(ck, r.config)
    except Exception as ex:
        FAIL.append((r.path, str(ex)[:200])); print("  [skip]", r.config, r.seed, str(ex)[:120]); continue
    m = m.to(device).eval(); ARCH_OF[r.path] = arch
    if r.config not in META:
        META[r.config] = dict(params_M=float(count_params(m)), gflops=safe_flops(m), arch=arch[0], encoder=arch[1], kind=arch[2])
    ctx = dict(config=r.config, seed=r.seed)
    for split in ("val", "test", "bolivia"):
        rows += chip_counts(m, CACHE[split], dict(ctx, split=split), "zero")
    if r.config in REPORT:
        rows += chip_counts(m, CACHE["test"], dict(ctx, split="test"), "mean")
    if not equiv_done:         # exactness check vs the package's own evaluate()
        ref = evaluate(m, DataLoader(CACHE["test"], batch_size=8, shuffle=False, num_workers=0), LGMFConfig(), device)
        tmp = pd.DataFrame([x for x in rows if x["split"] == "test" and x["variant"] == "zero" and x["config"] == r.config and x["seed"] == r.seed])
        mine = pooled(tmp, ["regime"]).set_index("regime"); minef = pooled(tmp, ["regime"], flood=True).set_index("regime")
        for reg in REGS:
            assert abs(mine.loc[reg, "iou"] - ref[reg]["iou"]) < 1e-6, (reg, mine.loc[reg, "iou"], ref[reg]["iou"])
            if ref[reg].get("flood_only_iou") is not None:
                assert abs(minef.loc[reg, "iou"] - ref[reg]["flood_only_iou"]) < 1e-6, ("flood", reg)
        equiv_done = True; print("  per-chip pooling == lgmf.engine.evaluate  (assert passed on", r.config, r.seed, ")")
    print(f"  {r.config:28s} seed{r.seed}  done  ({(time.time()-t0)/60:.1f} min)")
    del m, ck
    if device.type == "cuda": torch.cuda.empty_cache()
PC = pd.DataFrame(rows)
PC.to_csv(f"{OUT}/per_chip_counts.csv.gz", index=False)
print(f"\nper_chip_counts.csv.gz: {len(PC):,} rows; failures: {len(FAIL)}")
for f in FAIL: print("  FAIL", f)
SUMMARY["checkpoints_evaluated"] = int(PC[["config", "seed"]].drop_duplicates().shape[0]); SUMMARY["rebuild_failures"] = FAIL

## 7. Does Table 2 reproduce? (test split, zero-fill = the code path used for the paper)

In [ ]:
PS  = pooled(PC, KEYS)
PSF = pooled(PC, KEYS, flood=True)
AG, AGF = across_seeds(PS), across_seeds(PSF)
def lookup(cfg, split, variant, regime, flood=False):
    a = (AGF if flood else AG); q = a[(a.config == cfg) & (a.split == split) & (a.variant == variant) & (a.regime == regime)]
    return None if q.empty else q.iloc[0].to_dict()
measured = {}
for cfg in REPORT:
    for met, reg in (("both", "both"), ("s1_only", "s1_only"), ("s2_only", "s2_only")):
        measured[(cfg, met)] = lookup(cfg, "test", "zero", reg)
    measured[(cfg, "flood")] = lookup(cfg, "test", "zero", "s1_only", flood=True)
T2 = compare_to_published(measured, PUBLISHED); display(T2); T2.to_csv(f"{OUT}/table2_reproduction.csv", index=False)
stc = T2.status.value_counts().to_dict(); SUMMARY["table2_status_counts"] = stc
_ev = T2[T2.status != "NOT EVALUATED"]
SUMMARY["table2_not_evaluated"] = sorted(T2[T2.status == "NOT EVALUATED"].config.unique().tolist())
SUMMARY["table2_reproduces"] = bool(len(_ev) and (_ev.status == "OK").all())     # judged on the rows we could evaluate
note(f"### Table 2 reproduction: {stc}")
# std convention: which ddof matches the published +/- ?
ok0 = np.nanmean([abs(r.std0 - r.pub_std) for r in T2.itertuples() if r.status != "NOT EVALUATED"])
ok1 = np.nanmean([abs(r.std1 - r.pub_std) for r in T2.itertuples() if r.status != "NOT EVALUATED"])
note(f"mean |measured std - published std|: ddof=0 -> {ok0:.4f},  ddof=1 -> {ok1:.4f}   (published convention should match the smaller)")
SUMMARY["std_gap_ddof0_vs_ddof1"] = [round(ok0, 5), round(ok1, 5)]
if SUMMARY["table2_reproduces"]:
    note("VERDICT: every evaluated Table-2 mean reproduces within rounding -> NB-18 (re-derivation) is NOT needed."
         + (f"  Still missing checkpoints for: {SUMMARY['table2_not_evaluated']} (attach the NB-final output)." if SUMMARY["table2_not_evaluated"] else ""))
else:
    note("VERDICT: some Table-2 entries do not reproduce -> inspect MISMATCH/CHECK rows before running NB-12; NB-18 may be needed.")

## 8. Table 4 provenance + full results matrix
Table 4 shows 0.795 joint IoU for the SegFormer-concat model, Table 2 shows 0.810. Hypothesis: Table 4 is **validation** while Table 2 is **test** (unlabelled). Below: every model on every split.

In [ ]:
MX = AG[AG.variant == "zero"].pivot_table(index=["config", "regime"], columns="split", values="mean").reindex(columns=["val", "test", "bolivia"])
MX = MX.reset_index(); MX.to_csv(f"{OUT}/all_results_matrix.csv", index=False)
display(MX[MX.regime == "both"].round(3))
sc = {sp: lookup("segformer_concat", sp, "zero", "both") for sp in ("val", "test", "bolivia")}
note("### Table 4 provenance (SegFormer concat, joint IoU)\n" + ", ".join(f"{k}={v['mean']:.3f}" for k, v in sc.items() if v) + "   [Table 4 prints 0.795; Table 2 prints 0.810]")
if sc["val"] and sc["test"]:
    near = min(("val", "test", "bolivia"), key=lambda k: abs(sc[k]["mean"] - 0.795) if sc[k] else 9)
    note(f"-> 0.795 is closest to the {near.upper()} split.")
    SUMMARY["table4_0795_matches_split"] = near

## 9. Bolivia (Section 4.7 currently gives no numbers)

In [ ]:
BO = AG[(AG.split == "bolivia") & (AG.variant == "zero")].copy()
BOF = AGF[(AGF.split == "bolivia") & (AGF.variant == "zero")].copy(); BOF["regime"] = BOF["regime"] + "_flood"
BOT = pd.concat([BO, BOF]).pivot_table(index="config", columns="regime", values=["mean", "std0"])
BOT = BOT.reindex([c for c in REPORT if c in BOT.index] + [c for c in BOT.index if c not in REPORT])
display(BOT.round(3)); BOT.to_csv(f"{OUT}/bolivia_results.csv")
pb = PS[(PS.split == "bolivia") & (PS.variant == "zero") & (PS.config == "segformer_concat")]
if len(pb): note("### Bolivia, segformer_concat per-seed IoU:\n" + pb.pivot_table(index="seed", columns="regime", values="iou").round(3).to_string())

## 10. Degenerate predictions (the FPN 0.110) and chips with no water (Figure 7's three 0.00-IoU chips)
Collapse = a model that predicts (almost) everything water, so IoU equals water prevalence. And: a chip with **no** water in the ground truth has per-chip IoU 0 by construction (TP=0), whatever the model does — which would explain three "IoU 0.00" panels.

In [ ]:
pr = PS[(PS.variant == "zero")].copy()
pr["prevalence"] = (pr.tp + pr.fn) / (pr.tp + pr.fp + pr.fn + pr.tn)
pr["pred_frac"]  = (pr.tp + pr.fp) / (pr.tp + pr.fp + pr.fn + pr.tn)
pr["collapse"] = [collapse_label(a, b, c, d) for a, b, c, d in zip(pr.tp, pr.fp, pr.fn, pr.tn)]
col = pr[pr.collapse != ""][["config", "seed", "split", "regime", "iou", "prevalence", "pred_frac", "collapse"]]
note(f"### Collapsed (degenerate) runs: {len(col)}")
display(col.round(3) if len(col) else "none")
fpn = pr[pr.config.str.contains("fpn", case=False)]
if len(fpn):
    note("FPN rows (IoU vs water prevalence):")
    display(fpn[fpn.split == "test"][["config", "seed", "regime", "iou", "prevalence", "pred_frac", "collapse"]].round(3))
else:
    note("No FPN checkpoint found among attached datasets -> cannot confirm the 0.110 collapse; re-run FPN in NB-12 (or drop it from the paper).")
SUMMARY["collapsed_runs"] = col.round(4).to_dict("records")

# chips with no valid water in the label
tst = PC[(PC.split == "test") & (PC.variant == "zero") & (PC.regime == "both") & (PC.config == "segformer_concat") & (PC.seed == PC[PC.config == "segformer_concat"].seed.min())]
nowater = tst[(tst.tp + tst.fn) == 0]
note(f"### Test chips with ZERO water pixels in the ground truth: {len(nowater)} of {len(tst)} -> per-chip IoU is 0 by construction: {nowater.id.tolist()}")
SUMMARY["test_chips_without_water"] = nowater.id.tolist()
pcs = PC[(PC.split == "test") & (PC.variant == "zero") & (PC.config == "segformer_concat") & (PC.regime == "s1_only")].copy()
pcs["gt_water_px"] = pcs.tp + pcs.fn; pcs["pred_px"] = pcs.tp + pcs.fp
pcs["chip_iou"] = pcs.tp / (pcs.tp + pcs.fp + pcs.fn + EPS)
worst = pcs.groupby("id").agg(event=("event", "first"), gt_water_px=("gt_water_px", "mean"), pred_px=("pred_px", "mean"), chip_iou=("chip_iou", "mean")).sort_values("chip_iou").head(12)
note("Lowest per-chip IoU, segformer_concat, SAR-only, test (mean over seeds):"); display(worst.round(3))
worst.to_csv(f"{OUT}/worst_chips_sar_only_test.csv")

## 11. Evaluation-device bug in `flops_g` (Reviewer 2.11: "apparent device mismatch")

In [ ]:
if device.type != "cuda":
    note("device-mismatch test skipped (no GPU in this session)")
else:
    _cfg0 = CK[CK.config.isin(REPORT)].iloc[0]
    _m, _ = build_from_ckpt(torch.load(_cfg0.path, map_location="cpu", weights_only=False), _cfg0.config)
    _m = _m.to(device).eval(); before = next(_m.parameters()).device
    _ = flops_g(_m, LGMFConfig(), device="cpu"); after = next(_m.parameters()).device
    err = ""
    try:
        with torch.no_grad(): _m(torch.randn(1, 2, 64, 64, device=device), torch.randn(1, 6, 64, 64, device=device), torch.ones(1, 2, device=device))
    except Exception as ex: err = str(ex)[:140]
    note(f"### flops_g side effect\nmodel device before: {before} | after flops_g(..., 'cpu'): {after} | next CUDA forward: {'FAILS -> ' + err if err else 'ok'}")
    SUMMARY["flops_g_moves_model_inplace"] = bool(str(before) != str(after))
    del _m

## 12. Mean-fill vs zero-fill at inference (test split, paired per seed)
Models were **trained** with the code's fill; this swaps only the inference-time fill, to quantify how much the text/code mismatch could matter.
If Δ is ~0 (e.g. few non-finite pixels in test) the cheap fix is to correct the sentence in §3.2 and say so; if Δ is material, we need a retrain with true mean-fill (extra work package).

In [ ]:
pv = PS[(PS.split == "test") & PS.config.isin(REPORT)].pivot_table(index=["config", "seed", "regime"], columns="variant", values="iou").reset_index()
pv["delta_mean_minus_zero"] = pv["mean"] - pv["zero"]
dm = pv.groupby(["config", "regime"]).delta_mean_minus_zero.agg(["mean", "std", "min", "max"]).round(4)
display(dm); dm.to_csv(f"{OUT}/imputation_effect_test.csv")
aff = IMP[(IMP.split == "test")]
note(f"### Mean-fill vs zero-fill (test)\ntest chips with any non-finite S1 pixel: {int((aff.s1_nonfinite>0).sum())}/{len(aff)}; "
     f"max |delta IoU| over configs/regimes/seeds = {pv.delta_mean_minus_zero.abs().max():.4f}")
SUMMARY["imputation_max_abs_delta_iou_test"] = float(pv.delta_mean_minus_zero.abs().max())

## 13. Curriculum exposure (for the matched-exposure control, WP7) — code vs closed form

In [ ]:
cfg45 = LGMFConfig(); cfg45.epochs = 45
code_probs = np.array([regime_probs(cfg45, e) for e in range(45)])
form_mean, form_last = exposure(45, cfg45.warmup_epochs, cfg45.final_regime_probs)
assert np.allclose(code_probs.mean(0), form_mean, atol=1e-9)
note("### Curriculum (45 epochs, warm-up 5, terminal (0.4, 0.45, 0.15))\n"
     f"last-epoch probabilities (joint, SAR, opt): {np.round(code_probs[-1], 4).tolist()}\n"
     f"cumulative exposure (matched-exposure fixed probs for WP7): {np.round(code_probs.mean(0), 4).tolist()}")
SUMMARY["curriculum_last_epoch"] = np.round(code_probs[-1], 4).tolist(); SUMMARY["curriculum_cumulative_exposure"] = np.round(code_probs.mean(0), 4).tolist()

## 14. Release bundle (Editor: code in a DOI repository; checkpoints + split files requested by Reviewer 2.11)
Builds `release/` for the Zenodo deposit. The DOI itself is created by you on zenodo.org — nothing here invents one.

In [ ]:
def sha256(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()

shutil.copy(ns[0], f"{REL}/norm_stats.json")
if os.path.exists(f"{REL}/code"): shutil.rmtree(f"{REL}/code")
shutil.copytree(PKG_DIR, f"{REL}/code/lgmf", ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
code_hashes = {os.path.relpath(p, f"{REL}/code"): sha256(p) for p in sorted(glob.glob(f"{REL}/code/lgmf/**/*.py", recursive=True))}
json.dump(code_hashes, open(f"{REL}/code_sha256.json", "w"), indent=1)

man = []
os.makedirs(f"{REL}/weights", exist_ok=True)
for r in CK.itertuples():
    st = ck_val = None
    row = dict(config=r.config, seed=r.seed, source_path=r.path, size_mb=round(os.path.getsize(r.path) / 1e6, 2), sha256=sha256(r.path))
    if r.config in REPORT and r.path in ARCH_OF:
        ck = torch.load(r.path, map_location="cpu", weights_only=False)
        a, e, kind = ARCH_OF[r.path]
        os.makedirs(f"{REL}/weights/{r.config}", exist_ok=True)
        wp = f"{REL}/weights/{r.config}/seed{r.seed}.pt"
        torch.save({"state_dict": ck.get("ema") or ck["model"], "weights": "EMA", "config": r.config, "seed": r.seed,
                    "arch": a, "encoder": e, "kind": kind, "in_ch": 8, "val_at_selection": ck.get("val")}, wp)
        row.update(release_weights=os.path.relpath(wp, REL), weights_sha256=sha256(wp)); del ck
    man.append(row)
pd.DataFrame(man).to_csv(f"{REL}/checkpoints_manifest.csv", index=False)
json.dump({"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
           "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
           "numpy": np.__version__, "pandas": pd.__version__, "rasterio": rasterio.__version__, "lgmf": lgmf.__version__,
           "smp": __import__("segmentation_models_pytorch").__version__, "cudnn_benchmark": torch.backends.cudnn.benchmark,
           "model_params_M_gflops": META}, open(f"{REL}/env.json", "w"), indent=1, default=str)
open(f"{REL}/README_ZENODO_DRAFT.md", "w").write("""# Curriculum modality dropout and self-consistency for optical-SAR flood mapping with missing sensors

Code, split manifest and trained weights accompanying the Scientific Reports manuscript.

DOI: <assigned by Zenodo on deposit -- paste here and in the manuscript's Code Availability statement>

## Contents
- `code/lgmf/` -- training / evaluation package (sha256 of every file in `code_sha256.json`)
- `split_manifest.csv` -- chip id, event, partition (train/val/test/bolivia), footprint
- `norm_stats.json` -- per-band mean/std computed on the training split only
- `checkpoints_manifest.csv` -- every evaluated checkpoint with sha256
- `weights/` -- EMA weights for the models in Table 2 (3 seeds each: 42, 1, 2)
- `env.json` -- library versions and per-model parameter counts / GFLOPs
- `per_chip_counts.csv.gz` -- TP/FP/FN/TN per chip, regime, model, seed and split (all reported IoUs are pooled from these)

## Data
Sen1Floods11 (hand-labelled subset) -- obtain from the original authors; not redistributed here.

## Reproduce
Evaluation: load a weights file with `lgmf.baselines.build_baseline(arch, encoder, in_ch=8, pretrained=False)` and `load_state_dict(state_dict)`;
evaluate with `lgmf.engine.evaluate` (threshold 0.5, pooled confusion counts, no-data pixels masked).
""")
print("release/ written:"); os.system(f"du -sh {REL}/* | sort -h")

## 15. Report & verdict

In [ ]:
SUMMARY["environment"] = dict(torch=torch.__version__, lgmf=lgmf.__version__, gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else None)
SUMMARY["model_meta"] = META
json.dump(SUMMARY, open(f"{OUT}/audit_summary.json", "w"), indent=1, default=str)
open(f"{OUT}/audit_report.md", "w").write("# NB-11 audit report\n\n" + "\n\n".join(REPORT_LINES))
print("=" * 78)
print("NB-11 VERDICT")
print("=" * 78)
print(f"1. Table 2 reproduces from our checkpoints : {SUMMARY.get('table2_reproduces')}  {SUMMARY.get('table2_status_counts')}")
print(f"2. Split sizes match the paper               : {SUMMARY.get('split_sizes_match_paper')}  {SUMMARY.get('split_sizes')}")
print(f"3. Events shared across train/val/test       : {SUMMARY.get('events_shared_train_val_test')}/{SUMMARY.get('events_total')}")
print(f"4. Test chips touching a training chip       : {SUMMARY.get('adjacency_to_train', {}).get('test')}")
print(f"5. Max |dIoU| mean-fill vs zero-fill (test)  : {SUMMARY.get('imputation_max_abs_delta_iou_test'):.4f}")
print(f"6. flops_g moves model in place (bug)        : {SUMMARY.get('flops_g_moves_model_inplace')}")
print(f"7. Table-4 0.795 matches split               : {SUMMARY.get('table4_0795_matches_split')}")
print(f"8. Collapsed runs                            : {len(SUMMARY.get('collapsed_runs', []))}")
print(f"9. Test chips with no water (Fig. 7 cause?)  : {len(SUMMARY.get('test_chips_without_water', []))}")
print("\nNEXT: save this version, publish /kaggle/working as dataset 'lgmf-nb11-audit', then send me audit_report.md + table2_reproduction.csv.")